In [1]:
%useLatestDescriptors
%use dataframe
@file:DependsOn("com.github.doyaaaaaken:kotlin-csv-jvm:1.7.0")

import com.github.doyaaaaaken.kotlincsv.dsl.csvWriter
import io.github.oshai.kotlinlogging.KotlinLogging.logger
import kotlin.reflect.full.declaredMemberProperties

fun ensureFileExists(fileName: String): Boolean {
    val currentDir = System.getProperty("user.dir")
    val file = File(currentDir, fileName)

    if (!file.exists()) {
        file.createNewFile()
        return true
    }
    return false
}

private fun ensureFolderExists(fullPath: String) {
    val currentDir = System.getProperty("user.dir")
    val folder = File(fullPath)

    if (!folder.exists()) {
        folder.mkdirs()
    }
}

fun writeComparingCsv(headers: List<String>, instances: List<String>, revenues: List<List<Int>>, bestInstance: List<String>, fullPath: String) {
    ensureFolderExists(fullPath)

    csvWriter().open("$fullPath/comparison.csv") {
        writeRow(headers)
        instances.forEachIndexed{index, instance ->
            writeRow(listOf(instance) + revenues[index].map { it.toString() } + listOf(bestInstance[index]))
        }
    }

    println("CSV written successfully to $fullPath/comparison.csv")
}

fun <T : Any> writeCsv(data: List<T>, fileName: String, relativePath: String) {
    if (data.isEmpty()) {
        println("No data to write.")
        return
    }

    ensureFolderExists(relativePath)

    val kClass = data.first()::class
    val headers = kClass.declaredMemberProperties.map { it.name }

    csvWriter().open("$relativePath$fileName") {
        writeRow(headers)

        data.forEach { item ->
            val row = kClass.declaredMemberProperties.map { prop ->
                prop.getter.call(item)?.toString()?.replace(",", "\\,") ?: ""
            }
            writeRow(row)
        }
    }

    println("CSV written successfully to $relativePath$fileName")
}


fun compareTwoColumns (col1: DataColumn<*>, col2: DataColumn<*>, name: String): BaseColumn<String> {
    val col = col1.mapIndexed { index, value->
        val refValue = value as Int
        val compValue = col2[index] as Int
        if (refValue <= compValue) {
            (((compValue.toDouble() / refValue.toDouble()) - 1) * 100)
        }else {
            (((refValue.toDouble() / compValue.toDouble()) - 1) * -100)
        }.toInt().toString() + "\\%"
    }
    return col.rename(name)
}

private fun findDifferingSubstring(strings: List<String>): List<String> {
    val split = strings.map { it.split("_") }
    val transposed = split[0].indices.map { i -> split.map { it[i] } }

    val differingIndices = transposed
        .mapIndexedNotNull { index, parts ->
            if (parts.distinct().size > 1) index else null
        }

    return split.map { parts ->
        differingIndices.joinToString("_") { parts[it] }
    }
}
inline fun <reified T> transpose(xs: List<List<T>>): List<List<T>> {
    val cols = xs[0].size
    val rows = xs.size
    return List(cols) { j ->
        List(rows) { i ->
            xs[i][j]
        }
    }
}

fun mergeResults(fullpath: String) {
    val folder = File(fullpath)
    val files =  folder.listFiles()
        ?.filter { it.isFile && it.name != "comparison.csv"}
        ?: emptyList()
    val fileNames = findDifferingSubstring(files.map { it.nameWithoutExtension })

    val results = files.mapIndexed { index, file ->
        fileNames[index] to DataFrame.read(file)
    }
    val instances = results.first().second["name"].map { it.toString() }

    val bestAvgValues = instances.mapIndexed { index, _ ->
        val avgValues = results.map { it.second["revenueAvg"][index] as Int }
        val best = avgValues.maxOrNull()
        val bestIndex = avgValues.indexOf(best)
        results[bestIndex].first
    }
    val avgColumns = results.map {it.first to it.second["revenueAvg"] }

    writeComparingCsv(
        headers = listOf("instance") + fileNames + listOf("best"),
        instances = instances.toList(),
        revenues = transpose(avgColumns.map { it.second.toList().map { value -> value as Int} }),
        bestInstance = bestAvgValues.toList(),
        fullPath = fullpath
    )
}


In [2]:
// static stuff
enum class Context {CLUSTER, BUDGET, CLUSTERKM, CLUSTERALL, ELIMINATION}
enum class Mode {FLAT, RANDOM}
val percentageFraction = 1

val orderedInstances = listOf("eil101", "gil262", "pr299", "lin318", "rd400", "d493", "u574", "u724", "pcb1173", "fl1400", "pr2392").map { if (it == "instance") it else it + "-gen3-50" }

val mode = Mode.FLAT
val context = Context.CLUSTERALL


val baseline = "fbckmn"
val colsWithoutPercentages = when (context) {
    Context.CLUSTER -> baseline
    Context.BUDGET -> baseline
    Context.CLUSTERKM -> "kmn"
    Context.CLUSTERALL-> baseline
    Context.ELIMINATION -> "OP"
}

val colsToIgnoreWhenCalcuateMax = when (context) {
    Context.CLUSTER -> emptyList()
    Context.BUDGET -> emptyList()
    Context.CLUSTERKM -> emptyList()
    Context.CLUSTERALL -> listOf("kmn", "kmd")
    Context.ELIMINATION -> emptyList()
}

val relativePath = when (context) {
    Context.ELIMINATION -> "/op-solver-strict/results/elimination"
    Context.CLUSTERALL -> "/op-solver-strict/results/cluster/all${mode.name.lowercase()}"
    else -> "/op-solver-strict/results/${context.name.lowercase()}${mode.name.lowercase()}"
}
val header = when (context) {
    Context.CLUSTER -> listOf("instance", "rkmn", "rkmd", "nckmn", "nckmd", "fbckmn", "fbckmd", "best")
    Context.CLUSTERKM -> listOf("instance", "kmn", "kmd", "nckmn", "nckmd", "best")
    Context.CLUSTERALL -> listOf("instance", "rkmn", "rkmd", "nckmn", "nckmd", "fbckmn", "fbckmd", "kmn", "kmd", "best")
    Context.ELIMINATION -> listOf("instance","TSPrfb","TSPrce","OP")
    else -> null
}
data class OneRun(
    val budget: Int,
    val budgetSpentAvg: Double,
    val name: String,
    val revenueAvg: Int,
    val revenueMax: Int,
    val revenueMin: Int,
    val size: Int,
    val successfulAmount: Int,
    val timeAvg: Double,
    val timeMax: Double,
    val timeMin: Double
)

data class ClusterAll(
    val instance: String,
    val rkmn: Int,
    val rkmd: Int,
    val nckmn: Int,
    val nckmd: Int,
    val fbckmn: Int,
    val fbckmd: Int,
    val kmn: Int,
    val kmd: Int,
    val best: String
)

val label = "${context.toString().lowercase()}_${mode.toString().lowercase()}"

val caption = when (context) {
    Context.CLUSTER -> "Comparison of the clustering results for the instances in the ${mode.name.lowercase()} mode."
    Context.BUDGET -> "Comparison of the budget results for the instances in the ${mode.name.lowercase()} mode."
    Context.CLUSTERKM -> "Comparison of the k-means impact results for the instances in the ${mode.name.lowercase()} mode."
    Context.CLUSTERALL -> "Clustering Algorithm Comparison for ${mode.name.lowercase()} Instances. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$. The highest revenue of an instance has 100\\% saturation decreasing to 0\\% at 90\\% of the maximum revenue. The star refers to the best mean revenue (kmn and kmd excluded)."
    Context.ELIMINATION -> "Comparison of the elimination methods for ${mode.name.lowercase()} the instances. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$. The highest revenue of an instance has 100\\% saturation decreasing to 0\\% at 90\\% of the maximum revenue. The star refers to the best mean revenue."
}

val title = when (context) {
    Context.CLUSTER -> "clustering"
    Context.BUDGET -> "budget"
    Context.CLUSTERKM -> "k-means impact"
    Context.CLUSTERALL -> "clustering"
    Context.ELIMINATION -> "elimination comparison"
} + " ${mode.name.lowercase()}"

In [3]:
val pathToFolder  = java.nio.file.Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath()
.toString() + relativePath

val mainPath = java.nio.file.Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath()
    .toString() + relativePath + "/comparison.csv"

if (!File(mainPath).exists()) {
    mergeResults(pathToFolder)
}

var df = DataFrame.readCsv(mainPath).cast<ClusterAll>()
df = df.sortWith (compareBy { row -> orderedInstances.indexOf(row["instance"].toString()) })
    .reorderColumnsBy { colums -> header.indexOf(colums.name()) }
df


instance,rkmn,rkmd,nckmn,nckmd,fbckmn,fbckmd,kmn,kmd,best
eil101,58,58,56,57,59,59,59,59,kmd
gil262,137,135,130,128,137,139,139,139,kmd
pr299,154,150,147,144,150,150,151,157,kmd
lin318,183,180,173,173,183,188,193,183,kmn
rd400,204,200,194,192,192,207,209,207,kmn
d493,285,283,257,256,302,289,299,292,fbckmn
u574,303,301,304,295,309,312,243,316,kmd
u724,375,381,372,363,384,388,391,385,kmn
pcb1173,563,555,472,544,574,576,578,587,kmd
fl1400,960,923,789,799,984,966,1008,989,kmn


In [4]:
val referencePath = java.nio.file.Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath()
    .toString() + "/op-solver-strict/results/baseline_${mode.name.lowercase()}.csv"
val baselineDf = DataFrame.readCsv(referencePath).cast<OneRun>()
    .sortWith (compareBy { row -> orderedInstances.indexOf(row["name"].toString()) })

//baselineDf.sortBy { it["revenueAvg"] }.name.values().map { it.split("-").first() }.map { "\"$it\"" }

baselineDf

budget,budgetSpentAvg,name,revenueAvg,revenueMax,revenueMin,size,successfulAmount,timeAvg,timeMax,timeMin
315,305.998000,eil101,59,61,59,101,5,0.850000,1.290000,0.500000
1189,1154.836000,gil262,139,143,135,262,5,12.710000,20.930000,9.030000
24096,23463.412000,pr299,150,161,135,299,5,10.950000,29.530000,3.370000
21015,20550.542000,lin318,188,192,187,318,5,10.230000,16.390000,6.260000
7641,7474.072000,rd400,207,211,205,400,5,13.310000,18.880000,9.260000
17501,17145.658000,d493,289,307,273,493,5,28.910000,73.260000,14.350000
18453,17983.646000,u574,312,321,297,574,5,13.060000,21.890000,10.020000
20955,20481.636000,u724,388,392,383,724,5,36.480000,76.960000,19.050000
28446,27721.012000,pcb1173,576,586,558,1173,5,19.330000,26.950000,14.550000
10064,9794.830000,fl1400,966,1041,931,1400,5,85.700000,137.240000,70.400000


In [5]:
fun getLatexTable(formating: String, amountColumns: String, title: String, header: String, label: String, caption: String, body: String): String {
    return """
    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ $formating  }
                \hline
                \multicolumn{$amountColumns}{|c|}{$title} \\
                \hline
                    $header \\
                \hline
                    $body
                \hline
            \end{tabular}
        \end{adjustbox}
        \caption{$caption}
        \label{$label}
    \end{table}
         """
}

In [6]:
df = df.remove("best")
df.update("instance").with { it.toString().split("-")[0] }
val amountColumns = df.columns().size.toString()
val formating = "|"+ df.columns().joinToString(separator = "") { "p{1.7cm}|" }
val header = df.columnNames().joinToString(separator = " & ")

df

instance,rkmn,rkmd,nckmn,nckmd,fbckmn,fbckmd,kmn,kmd
eil101,58,58,56,57,59,59,59,59
gil262,137,135,130,128,137,139,139,139
pr299,154,150,147,144,150,150,151,157
lin318,183,180,173,173,183,188,193,183
rd400,204,200,194,192,192,207,209,207
d493,285,283,257,256,302,289,299,292
u574,303,301,304,295,309,312,243,316
u724,375,381,372,363,384,388,391,385
pcb1173,563,555,472,544,574,576,578,587
fl1400,960,923,789,799,984,966,1008,989


In [7]:
val bestValues = df.convert { all()}.perRowCol { row, col ->
    if (col[row] is String || colsToIgnoreWhenCalcuateMax.contains(col.name())) {
        0
    } else {
        col[row] as Int
    }
}.map{ row ->
    row.rowMaxOf<Int>()
}

bestValues


[59, 139, 154, 188, 207, 302, 312, 388, 576, 984, 1212]

In [8]:


val rowMaxValues = bestValues.mapIndexed { index, resultMax -> max(resultMax as Int, baselineDf["revenueAvg"][index] as Int) }

val rowMinValues = df.map { row ->
    row.rowMinOf<Int>()
}

val revenueDif = rowMaxValues.mapIndexed { index, maxEntry ->
    val minEntry = df[index].rowMinOf<Int>()
    (maxEntry - minEntry).toDouble() / maxEntry.toDouble()
}
val maxRevenueDif = revenueDif.max()
val gradient = 0.1 //max(maxRevenueDif,0.0)

fun getSaturation (gradient: Double, maxValue: Int, value: Int): String {
    return min(max((100-((maxValue - value) / (maxValue * gradient) * 100)),0.0),100.0).toInt().toString()
}

rowMaxValues


[59, 139, 154, 188, 207, 302, 312, 388, 576, 984, 1212]

In [9]:
import java.util.Locale

fun calculatePercentage(refValue: Int, compValue: Int): Double{ return ((compValue.toDouble() - refValue.toDouble()) / refValue.toDouble())}
fun formatePercentage(value: Double): String { return "${String.format(Locale.US, "%+.${percentageFraction}f", value * 100)}\\%" }

fun getPercentage(row: DataRow<*>, compValue: Int): String {
    val refValue = df.get(colsWithoutPercentages)[row] as Int
    val percentage = calculatePercentage(refValue, compValue)
    return "{\\tiny${formatePercentage(percentage)}}"
}

val footer = df.convert { all() }.perRowCol { row, col ->
    if (col.name() == colsWithoutPercentages || col[row] is String) {
        10000.0
    } else {
        val refValue = df.get(colsWithoutPercentages)[row] as Int
        calculatePercentage(refValue, col[row] as Int)
    }
}.mean().values().mapIndexed { index, it ->
    if (index == 0) {
        "avg diff"
    } else if (it is Double && it > 100.0) {
        "-"
    } else if (it is Double) {
        formatePercentage(it)
    } else {
        "${it.toString()}\\%"
    }
}.toList()
footer

[avg diff, -0.9\%, -2.2\%, -7.3\%, -6.8\%, -, +0.6\%, -0.2\%, +1.4\%]

In [10]:
val stringdf = df.convert { all() }.perRowCol { row, col  ->
    if (col[row] is String) {
        col[row].toString().split("-").first()
    } else if (col.name() == colsWithoutPercentages) {
        val value = col[row] as Int
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        if (bestValues[row.index()] == value && !colsToIgnoreWhenCalcuateMax.contains(col.name())){
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}"
        }else {
            "\\cellcolor{cyan!$saturation} $value"
        }
    } else {
        val value = col[row] as Int
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        val percentage = getPercentage(row, value)
        if (bestValues[row.index()] == value && !colsToIgnoreWhenCalcuateMax.contains(col.name())) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}$percentage"
        }else {
            "\\cellcolor{cyan!$saturation} $value$percentage"
        }
    }
}
stringdf

instance,rkmn,rkmd,nckmn,nckmd,fbckmn,fbckmd,kmn,kmd
eil101,\cellcolor{cyan!83} 58{\tiny-1.7\%},\cellcolor{cyan!83} 58{\tiny-1.7\%},\cellcolor{cyan!49} 56{\tiny-5.1\%},\cellcolor{cyan!66} 57{\tiny-3.4\%},\cellcolor{cyan!100} \textbf{59*},\cellcolor{cyan!100} \textbf{59*}{\ti...,\cellcolor{cyan!100} 59{\tiny+0.0\%},\cellcolor{cyan!100} 59{\tiny+0.0\%}
gil262,\cellcolor{cyan!85} 137{\tiny+0.0\%},\cellcolor{cyan!71} 135{\tiny-1.5\%},\cellcolor{cyan!35} 130{\tiny-5.1\%},\cellcolor{cyan!20} 128{\tiny-6.6\%},\cellcolor{cyan!85} 137,\cellcolor{cyan!100} \textbf{139*}{\t...,\cellcolor{cyan!100} 139{\tiny+1.5\%},\cellcolor{cyan!100} 139{\tiny+1.5\%}
pr299,\cellcolor{cyan!100} \textbf{154*}{\t...,\cellcolor{cyan!74} 150{\tiny+0.0\%},\cellcolor{cyan!54} 147{\tiny-2.0\%},\cellcolor{cyan!35} 144{\tiny-4.0\%},\cellcolor{cyan!74} 150,\cellcolor{cyan!74} 150{\tiny+0.0\%},\cellcolor{cyan!80} 151{\tiny+0.7\%},\cellcolor{cyan!100} 157{\tiny+4.7\%}
lin318,\cellcolor{cyan!73} 183{\tiny+0.0\%},\cellcolor{cyan!57} 180{\tiny-1.6\%},\cellcolor{cyan!20} 173{\tiny-5.5\%},\cellcolor{cyan!20} 173{\tiny-5.5\%},\cellcolor{cyan!73} 183,\cellcolor{cyan!100} \textbf{188*}{\t...,\cellcolor{cyan!100} 193{\tiny+5.5\%},\cellcolor{cyan!73} 183{\tiny+0.0\%}
rd400,\cellcolor{cyan!85} 204{\tiny+6.3\%},\cellcolor{cyan!66} 200{\tiny+4.2\%},\cellcolor{cyan!37} 194{\tiny+1.0\%},\cellcolor{cyan!27} 192{\tiny+0.0\%},\cellcolor{cyan!27} 192,\cellcolor{cyan!100} \textbf{207*}{\t...,\cellcolor{cyan!100} 209{\tiny+8.9\%},\cellcolor{cyan!100} 207{\tiny+7.8\%}
d493,\cellcolor{cyan!43} 285{\tiny-5.6\%},\cellcolor{cyan!37} 283{\tiny-6.3\%},\cellcolor{cyan!0} 257{\tiny-14.9\%},\cellcolor{cyan!0} 256{\tiny-15.2\%},\cellcolor{cyan!100} \textbf{302*},\cellcolor{cyan!56} 289{\tiny-4.3\%},\cellcolor{cyan!90} 299{\tiny-1.0\%},\cellcolor{cyan!66} 292{\tiny-3.3\%}
u574,\cellcolor{cyan!71} 303{\tiny-1.9\%},\cellcolor{cyan!64} 301{\tiny-2.6\%},\cellcolor{cyan!74} 304{\tiny-1.6\%},\cellcolor{cyan!45} 295{\tiny-4.5\%},\cellcolor{cyan!90} 309,\cellcolor{cyan!100} \textbf{312*}{\t...,\cellcolor{cyan!0} 243{\tiny-21.4\%},\cellcolor{cyan!100} 316{\tiny+2.3\%}
u724,\cellcolor{cyan!66} 375{\tiny-2.3\%},\cellcolor{cyan!81} 381{\tiny-0.8\%},\cellcolor{cyan!58} 372{\tiny-3.1\%},\cellcolor{cyan!35} 363{\tiny-5.5\%},\cellcolor{cyan!89} 384,\cellcolor{cyan!100} \textbf{388*}{\t...,\cellcolor{cyan!100} 391{\tiny+1.8\%},\cellcolor{cyan!92} 385{\tiny+0.3\%}
pcb1173,\cellcolor{cyan!77} 563{\tiny-1.9\%},\cellcolor{cyan!63} 555{\tiny-3.3\%},\cellcolor{cyan!0} 472{\tiny-17.8\%},\cellcolor{cyan!44} 544{\tiny-5.2\%},\cellcolor{cyan!96} 574,\cellcolor{cyan!100} \textbf{576*}{\t...,\cellcolor{cyan!100} 578{\tiny+0.7\%},\cellcolor{cyan!100} 587{\tiny+2.3\%}
fl1400,\cellcolor{cyan!75} 960{\tiny-2.4\%},\cellcolor{cyan!38} 923{\tiny-6.2\%},\cellcolor{cyan!0} 789{\tiny-19.8\%},\cellcolor{cyan!0} 799{\tiny-18.8\%},\cellcolor{cyan!100} \textbf{984*},\cellcolor{cyan!81} 966{\tiny-1.8\%},\cellcolor{cyan!100} 1008{\tiny+2.4\%},\cellcolor{cyan!100} 989{\tiny+0.5\%}


In [11]:
val body = stringdf.rows().joinToString(separator = " \\\\ \n") { row ->
    row.values().joinToString(separator = " & ")  {
        it.toString()
    }
} + " \\\\ \\hline " + footer.joinToString(separator = " & ")  {
    it.toString()
} + " \\\\"

getLatexTable(formating, amountColumns, title, header, label, caption, body)


    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ |p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|p{1.7cm}|  }
                \hline
                \multicolumn{9}{|c|}{clustering flat} \\
                \hline
                    instance & rkmn & rkmd & nckmn & nckmd & fbckmn & fbckmd & kmn & kmd \\
                \hline
                    eil101 & \cellcolor{cyan!83} 58{\tiny-1.7\%} & \cellcolor{cyan!83} 58{\tiny-1.7\%} & \cellcolor{cyan!49} 56{\tiny-5.1\%} & \cellcolor{cyan!66} 57{\tiny-3.4\%} & \cellcolor{cyan!100} \textbf{59*} & \cellcolor{cyan!100} \textbf{59*}{\tiny+0.0\%} & \cellcolor{cyan!100} 59{\tiny+0.0\%} & \cellcolor{cyan!100} 59{\tiny+0.0\%} \\ 
gil262 & \cellcolor{cyan!85} 137{\tiny+0.0\%} & \cellcolor{cyan!71} 135{\tiny-1.5\%} & \cellcolor{cyan!35} 130{\tiny-5.1\%} & \cellcolor{cyan!20} 128{\tiny-6.6\%} & \cellcolor{cyan!85} 137 & \cellcolor{cyan!100} \textbf{139*}{\tiny+1.5\%} & \cel